In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

# 1. GENERATE SYNTHETIC TRAINING DATA
np.random.seed(42)
num_samples = 1000

# Features: Income (in thousands), Daily Commute (miles)
incomes = np.random.uniform(30, 200, num_samples)
commutes = np.random.uniform(5, 120, num_samples)

# Targets: We logically map their features to what they would care about
price_weights = 1.0 / (incomes / 30)  # Lower income = higher price weight
range_weights = commutes / 120        # Longer commute = higher range weight
battery_weights = np.random.uniform(0.1, 0.3, num_samples) # Baseline battery preference

# Normalize the weights so they always equal 1.0 (100%)
totals = price_weights + range_weights + battery_weights
price_weights /= totals
range_weights /= totals
battery_weights /= totals

# Create a DataFrame
user_df = pd.DataFrame({
    'Income_k': incomes,
    'Commute_miles': commutes,
    'Target_Price_W': price_weights,
    'Target_Range_W': range_weights,
    'Target_Battery_W': battery_weights
})

print(f"✅ Generated {len(user_df)} synthetic user profiles for training!")
print(user_df.head())

✅ Generated 1000 synthetic user profiles for training!
     Income_k  Commute_miles  Target_Price_W  Target_Range_W  Target_Battery_W
0   93.671820      26.290287        0.463018        0.316738          0.220244
1  191.621432      67.318609        0.180587        0.647088          0.172325
2  154.438970     105.388771        0.143492        0.648750          0.207758
3  131.771942      89.205862        0.203100        0.663167          0.133733
4   56.523169      97.754532        0.353892        0.543165          0.102943


In [9]:
# 2. DEFINE THE NEURAL NETWORK
class PreferenceCalibrationNet(nn.Module):
    def __init__(self):
        super(PreferenceCalibrationNet, self).__init__()
        # Input layer (2 features) -> Hidden layer
        self.fc1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        # Hidden layer -> Output layer (3 weights)
        self.fc2 = nn.Linear(16, 3)
        # Softmax ensures the 3 outputs sum to 1.0 (like percentages)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

# Initialize the model
model = PreferenceCalibrationNet()
print("🧠 Neural Network Architecture Built:")
print(model)

🧠 Neural Network Architecture Built:
PreferenceCalibrationNet(
  (fc1): Linear(in_features=2, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=3, bias=True)
  (softmax): Softmax(dim=1)
)


In [10]:
# 3. PREPARE DATA FOR PYTORCH
# Convert our Pandas DataFrame columns into PyTorch Tensors
X = torch.tensor(user_df[['Income_k', 'Commute_miles']].values, dtype=torch.float32)
y = torch.tensor(user_df[['Target_Price_W', 'Target_Range_W', 'Target_Battery_W']].values, dtype=torch.float32)

print("📊 Tensors ready!")
print(f"X (Input Features) shape: {X.shape}")
print(f"y (Target Weights) shape: {y.shape}")

📊 Tensors ready!
X (Input Features) shape: torch.Size([1000, 2])
y (Target Weights) shape: torch.Size([1000, 3])


In [11]:
# 4. TRAIN THE NEURAL NETWORK
# We use Mean Squared Error to measure how far off our predicted weights are
criterion = nn.MSELoss()

# Adam optimizer acts as the "steer" to correct the weights
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 500
print("🚀 Starting Training Loop...")

for epoch in range(epochs):
    # 1. Forward pass: Make predictions for all 1000 users
    predictions = model(X)
    
    # 2. Calculate the loss (how wrong the predictions were)
    loss = criterion(predictions, y)
    
    # 3. Backward pass: Calculate gradients
    optimizer.zero_grad()
    loss.backward()
    
    # 4. Update the network's internal weights
    optimizer.step()
    
    # Print progress every 100 epochs to watch it learn
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}")

print("✅ Training Complete! The model has calibrated.")

🚀 Starting Training Loop...
Epoch [100/500] | Loss: 0.0097
Epoch [200/500] | Loss: 0.0058
Epoch [300/500] | Loss: 0.0033
Epoch [400/500] | Loss: 0.0018
Epoch [500/500] | Loss: 0.0018
✅ Training Complete! The model has calibrated.


In [12]:
# 5. TEST THE CALIBRATION MODEL ON NEW USERS

def predict_user_preferences(income_k, commute_miles, description):
    model.eval() # Set model to evaluation mode (turns off training)
    with torch.no_grad(): # We don't need to calculate gradients for predicting
        # Create a tensor for the new user
        user_tensor = torch.tensor([[income_k, commute_miles]], dtype=torch.float32)
        
        # Predict the weights!
        predicted_weights = model(user_tensor).numpy()[0]
        
        price_w, range_w, battery_w = predicted_weights
        
        print(f"👤 User Profile: {description}")
        print(f"   ↳ Income: ${income_k}k | Daily Commute: {commute_miles} miles")
        print("🤖 Neural Network Calibrated Preferences:")
        print(f"   ↳ 💰 Price Importance:   {price_w * 100:.1f}%")
        print(f"   ↳ 🛣️ Range Importance:   {range_w * 100:.1f}%")
        print(f"   ↳ 🔋 Battery Importance: {battery_w * 100:.1f}%\n")

# Test Case 1: A college student with a tight budget and short commute
predict_user_preferences(income_k=30.0, commute_miles=10.0, description="Budget-Conscious Student in Santa Cruz")

# Test Case 2: A wealthy executive with a massive super-commute
predict_user_preferences(income_k=190.0, commute_miles=110.0, description="Wealthy Super-Commuter")

👤 User Profile: Budget-Conscious Student in Santa Cruz
   ↳ Income: $30.0k | Daily Commute: 10.0 miles
🤖 Neural Network Calibrated Preferences:
   ↳ 💰 Price Importance:   65.5%
   ↳ 🛣️ Range Importance:   16.9%
   ↳ 🔋 Battery Importance: 17.5%

👤 User Profile: Wealthy Super-Commuter
   ↳ Income: $190.0k | Daily Commute: 110.0 miles
🤖 Neural Network Calibrated Preferences:
   ↳ 💰 Price Importance:   8.6%
   ↳ 🛣️ Range Importance:   76.5%
   ↳ 🔋 Battery Importance: 14.9%



In [13]:
# 6. EXPORT THE TRAINED MODEL
import os

# Create a folder to hold your saved models if it doesn't exist
os.makedirs('../saved_models', exist_ok=True)

# Define the file path
model_path = '../saved_models/preference_calibrator.pth'

# Save the model's internal weights and biases (its "memory")
torch.save(model.state_dict(), model_path)

print(f"💾 Success! Neural Network saved to: {model_path}")

💾 Success! Neural Network saved to: ../saved_models/preference_calibrator.pth
